# TIEGCM Parsl workflow

This notebook is the stand-alone companion to the Parsl workflow in `main.py` in this repository. This notebook is designed to be run directly on an HPC resource while the `main.py` in this workflow uses the `parsl_utils` to launch ensemble members from a central coordinating node (i.e. a laptop or the Parallel Works platform). This workflow simulates a typical TIEGCM data assimilating workflow with the following tasks:
1. configure Parsl and start Parsl monitoring to visualize workflow progress;
2. download TIEGCM container;
3. download and preprocess solar forcing;
4. set up siloed work directories for ensemble members;
5. launch and monitor the ensemble members;
6. post process the ensemble member output;
7. run data assimilation (or a DA placeholder); and
8. any clean up/restructuring necessary for running another DA cycle.

## Workflow visualization

There are three options for visualizing a Parsl workflow:
1. Manual "direct" launch of `parsl-visualize` before workflow runs;
2. Visualization launched as part of the workflow (i.e. self-launch); and
3. Offline visualization (i.e. for browsing previous Parsl workflows or starting visualization completely independently of the workflow).

Examples for all three are provided below but options \#1 and \#3 are commented out.

## Workflow parameters

The key customizable parameters for this workflow are defined immediately below. For a fully automated workflow (i.e. non-interactive workflow in `main.py`), these parameters are typically specified in the workflow launch form (and corresponding `.json` package for API launch) and then they make it to the command line launch of `main.py` on the head node of the cluster.

In [ ]:
# Workflow parameters
install=False
install_from_scratch=False
conda_base_path="~/pw/software/.miniconda3c/"
conda_env_name="parsl"

param_log_dir='./parsl-app-logs'
param_work_dir_root='./tiegcm-work'
# Prefix matches **singularity pull** => containers on DockerHub are prefixed with docker:// 
param_container_url='docker://parallelworks/tiegcm:latest'

## Installs

There are two install options:
1. `install_from_scratch = True` documents the steps to build a particular environment
2. `install_from_scratch = False` is faster to reconstruct a Conda environment from an exported env file `.yaml` than to rebuild from scratch and promotes reproducibility.

The reconstruction command is kept active here since 
env files are distributed with this notebook. Once the command to reconstruct
the Conda environment has been run, you may need to tell
this notebook to use the kernel from that Conda environment
with the `Kernel > Change kernel...` option in the menu above.

In [ ]:
# You don't need to rerun the install if the environment has already
# been built and selected as the kernel for this notebook.
if (install):
    if (install_from_scratch):
        
        # Currently there is a dependency bug with ipykernel and Python 3.13,
        # so pin to a different Python.
        ! conda create -y --name {conda_env_name} python=3.9
        
        # To use a Jupyter notebook with a
        # specific conda environment:
        ! conda install -y --name {conda_env_name} requests
        ! conda install -y --name {conda_env_name} ipykernel
        ! conda install -y --name {conda_env_name} -c anaconda jinja2
        
        # Additional packages for near-real time data streams:
        ! conda install -y --name {conda_env_name} psycopg2
        
        
        # pip installs
        # Conda does not install monitoring, so use pip
        # Each Conda env has its own pip, so need to activate.
        ! source {conda_base_path}/etc/profile.d/conda.sh; conda activate {conda_env_name}; pip install --upgrade pip
        ! source {conda_base_path}/etc/profile.d/conda.sh; conda activate {conda_env_name}; pip install 'parsl[monitoring, visualization]'
        
        # The environment was then exported with:
        ! conda env export --name {conda_env_name} > ./requirements/{conda_env_name}.yaml
    else:
        # You can rebuild the environment with:
        ! conda env update -f ./requirements/{conda_env_name}.yaml --name {conda_env_name}


## Imports

Based on the instructions in the [Parsl Tutorial](https://parsl.readthedocs.io/en/latest/1-parsl-introduction.html)

In [ ]:
import os
import numpy as np
#import pandas as pd

# parsl dependencies
import parsl
import logging
from parsl.app.app import python_app, bash_app
from parsl.configs.local_threads import Config
from parsl.executors import HighThroughputExecutor # We want to use monitoring, so we must use HTEX
from parsl.executors import MPIExecutor # MPIExecutor wraps the HTEX; need to test if can be used with monitoring
from parsl.monitoring.monitoring import MonitoringHub
from parsl.addresses import address_by_hostname
from parsl.providers import SlurmProvider, LocalProvider

# to display Parsl monitoring GUI in notebook
# Experimental - does not work yet
from IPython.display import IFrame

# To grab near real time solar intensity predictions
from fetch_indices import get_f107
from fetch_indices import get_kp_array

#=================================================
# Log everything to stdout (ends up in pink boxes 
# in the notebook). This information is logged anyway
# in ./runinfo/<run_id>/parsl.log. Careful - this has
# the potential to slow the notebook down significantly
# for complex workflows.
# parsl.set_stream_logger() # <-- log everything to stdout
#==================================================

print(parsl.__version__)

## Configure Parsl

This configuration must use the `HighThroughputExecutor` (HTEX) since we also want to enable [Parsl monitoring](https://parsl.readthedocs.io/en/latest/userguide/monitoring.html).

In [ ]:
config = Config(
    retries=3,
    executors=[
       # Use slurm_htex for running SLURM jobs on the worker nodes
       HighThroughputExecutor(
           label="slurm_htex",
           #cores_per_worker=1,
           #max_workers_per_node=2,
           address=address_by_hostname(),
           provider=SlurmProvider(
               partition='small',
               nodes_per_block=1,
               cores_per_node=4,
               init_blocks=3,
               min_blocks=3,
               max_blocks=10,
               exclusive=True,
               worker_init="source "+conda_base_path+"/etc/profile.d/conda.sh; conda activate "+conda_env_name,
               # I don't know why, but Parsl is refusing to relaunch
               # Parsl workers via SLURM. Perhaps the default walltime
               # of 30 mins is a frim, absolute, limit whereas in the
               # past I *think* it has been treated as a per-pilot job limit
               # and new workers would get called up as needed.
               walltime="10:00:00"
           )
           
       )
   ],
   monitoring=MonitoringHub(
       hub_address=address_by_hostname(),
       hub_port=55055,
       monitoring_debug=False,
       resource_monitoring_interval=10,
   ),
   strategy='none'
)

# Loading the configuration starts a Parsl DataFlowKernel
dfk = parsl.load(config)

## Define Parsl apps

Parsl workflows are divided into the smallest unit of execution, the app. There are two types of Parsl apps:
1. `python_app`s are useful when launching pure Python code. They are also particularly useful if you want to pass a *small* amount of output from a running app directly back into the workflow. For example, `make_dir` below could be set up as a `python_app` or a `bash_app`. But, I choose to make it a `python_app` because I want the value of `my_dir` to be made explicitly available to the workflow via the `.result()` of the app. This is not possible with a `bash_app`.
2. `bash_app`s are useful when launching tasks on the command line

Here, the applications are *defined* but not run. The `@python_app` and `@bash_app` decorators are the "flags" that tell Parsl that these functions are special and need to be tracked as part of the workflow. Undecorated functions execute locally as regular Python in whatever runtime the notebook is in.

### Python Apps

In [ ]:
@python_app # make directory to keep all files associated with the ML model
def make_dir(my_dir):
    import os
    os.makedirs(my_dir, exist_ok = False)
    return my_dir

### Bash Apps

In [ ]:
@bash_app # Start the parsl visulaizer
def start_parsl_visualize(
    stdout='parsl_vis_app.stdout', 
    stderr='parsl_vis_app.stderr'):
    return 'parsl-visualize --listen 127.0.0.1 --port 8080'

In [ ]:
@bash_app # Get TIEGCM container on-the-fly
def get_container(
    enforce,
    image_dir, 
    container_url, 
    stdout='get_tiegcm_container_app.stdout', 
    stderr='get_tiegcm_container_app.stderr'):
    return '''
    singularity pull {image_dir}/tiegcm.sif {container_url}
    '''.format(
        # I want the image_dir to be a dependency,
        # so it must be a Parsl Future. But the
        # path is stored as the .result() of that
        # Future, hence, the modification here.
        image_dir=image_dir,
        container_url=container_url)

In [ ]:
@bash_app # Test broadcasting jobs across the worker nodes
def worker_hello(enforce, stdout='worker_hello_app.stdout', stderr='worker_hello_app.stdout'):
    return '''
    date
    hn=`hostname`
    echo Host: $hn
    pwd
    whoami
    '''

@bash_app # Test multiple peer dependencies
def count_workers(inputs=(), log_dir="./logs", stdout='count_workers_app.stdout', stderr='count_workers_app.stdout'):
    return '''
    echo `cat {log_dir}/parsl_hello_app.stdout | col | grep Host | sort | uniq -c`
    '''.format(
        log_dir=log_dir
    )

## Start Parsl monitoring - Option 1 - direct shell invocation to background

This step can be done at any point provided that a database file exists.  The default location of this file is in `./runinfo/monitoring.db` and this file is created when the Parsl configuration is loaded. When the notebook kernel is restarted, additional Parsl workflow runs' information is appended to the monitoring information in `./runinfo`. It is possible to view this information "offline" (i.e. no active running Parsl workflows) and also fully independently of this notebook (see Option 3, at the end of this notebook for how to specify custom locations fort the monitoring DB). For our purposes here, we will assume that we're using a monitoring DB in the same directory as the notebook runtime in `./runinfo`.

This launch can be commented out here since it is also possible to launch `parsl-visualize` from a Parsl app within the workflow, examples of which are below. The advantage to running `parsl-visualize` as a Parsl app is that the visualization server is up and running while the workflow is running and then is shut down when the workflow is cleaned up. Otherwise, when `parsl-visualize` is launched via `os.system` the running child process can persist even after workflow shut down or notebook kernel restart. Here, however, we opt not to use `parsl-visualize` in the workflow because we want only one executor for this workflow for simplicity (to send jobs to the SLURM scheduler) but `parsl-visualize` would need to be started on a local (to the head node) executor.

You can rerun this command even if `parsl-visualize` is already running because it checks for existing port usage and if that port is already in use, it fails silently here (on the command line it will give you an error).

In [ ]:
# Launch Parsl 
os.system('parsl-visualize 1> parsl_vis.stdout 2> parsl_vis.stderr &')

## Parsl Workflow starts here

### Make a directory for workflow app logs

As part of the workflow, we make a directory for all the app logs. It looks like the first Parsl app to be invoked will start the Parsl interchanges and pilot jobs. This means that you may need to wait some time for this first app to start if you have a queue/cloud spinup wait time associated with getting an allocation of worker nodes **even if this first app is NOT going to worker nodes**!

In [ ]:
# Launch the app
make_log_dir_future = make_dir(param_log_dir)

# Get the result
log_dir = make_log_dir_future.result()

### Start Parsl monitoring - Option 2 - Monitoring as an app within a Parsl workflow

This approach is helpful if we want Parsl Monitoring processes to be cleaned up after the workflow is complete. Note that this command is tracked by Parsl and is considered to be part of the workflow. Since we defined the app to use the `local_htex` above, it is running on the head node of the cluster. You can verify this placement in the terminal with `ps -u $USER -HF -ww | grep vis`.

In [ ]:
# Start Parsl visualization in a
# separate cell since we only want
# to run this app one time.
#parsl_vis_future = start_parsl_visualize(
    #make_log_dir_future, 
#    stdout=log_dir+'/parsl_vis_app.stdout', 
#    stderr=log_dir+'/parsl_vis_app.stderr')

In [ ]:
# I'd love to view Parsl monitoring in the notebook,
# but this doesn't work.
# IFrame('http://localhost:8080', width=600, height=500)

### Set up the TIEGCM ensemble

Get container and establish working directories/parameters for each ensemble member. For development, it's often nice to separate each Parsl app in it's own cell so you can more easily see error messages. Otherwise, error messages can be obscured. Note that the `worker_hello` app is helpful for testing broadcasting an application to many nodes/workers. It is important to ensure that you make a **list** of app futures when launching many Parsl apps - if you overwrite a Parsl future of a still-running application, Parsl tends to get confused and blocks those applications in `launched` state but cannot proceed to update overwritten futures.

In [ ]:
# Example of launching 100 parallel tasks in Parsl
future_list=[]
for ii in np.linspace(1,100,100):
    future_list.append(
        worker_hello(
            make_log_dir_future,
            stdout=log_dir+'/parsl_hello_app.stdout',
            stderr=log_dir+'/parsl_hello_app.stderr'))
    

count_future = count_workers(
    inputs=future_list,
    log_dir=log_dir,
    stdout=log_dir+'/parsl_count_app.stdout',
    stderr=log_dir+'/parsl_count_app.stderr')

In [ ]:
make_work_dir_future = make_dir(param_work_dir_root)
work_dir=make_work_dir_future.result()

In [ ]:
# Get the container as a Parsl App
get_container_as_parsl_app=False
# This works, but it is *very* slow for a cluster without (expensive) 
# high performance storage because the job runs on a worker node
# which attempts to write the ~2GB container to an NFS-mounted
# shared drive that is on the head node. It's much faster to just
# run the singularity pull command on the head node. The alternative
# is to set up two executors, but that is not done here.
if (get_container_as_parsl_app):
    get_container_future = get_container(
        enforce=make_work_dir_future,
        image_dir=work_dir, 
        container_url=param_container_url, 
        stdout=log_dir+'/get_tiegcm_container_app.stdout', 
        stderr=log_dir+'/get_tiegcm_container_app.stderr')
else:
    # Local run of singularity pull (i.e. on head node)
    os.system('singularity pull '+work_dir+'/tiegcm.sif '+param_container_url+' 1> singularity_pull.stdout 2> singularity_pull.stderr &')

In [ ]:
get_container_future

### Get solar forcing data
```
# Import tar and uncompress
data_dir="tiegcm_res5.0_data"
if [ -d "$data_dir" ]; then
  echo "$data_dir exists."
else
    echo  Copying tar and uncompressing...
    #cp /storage/model-workflows/tiegcm/tiegcm2.0/data/tiegcm2.0_res5.0_data.tar.gz .
    cp /tiegcm/model-workflows-tiegcm2.0/data/tiegcm2.0_res5.0_data.tar.gz .

    tar -zxf tiegcm2.0_res5.0_data.tar.gz
```


### Add ancilliary run scripts
```
script_dir="script"
if [ -d "$script_dir" ]; then
  echo "$script_dir exists."
else
    echo  Copying tar and uncompressing...
    #cp -r /storage/model-workflows/tiegcm/tiegcm2.0/script .
    cp -r /tiegcm/model-workflows-tiegcm2.0/script .

    # TODO: Fix this workaround
    touch script/truncated_samples_F107.txt
```

### Create ensemble parameters
```
export TGCMMODEL=${PWD}
export TGCMDATA=${TGCMMODEL}/tiegcm_res5.0_data

# Load shared libraries installed using miniconda
export SINGULARITYENV_LD_LIBRARY_PATH=/opt/miniconda3/lib

# Choose ensemble size and other parameters for F10.7 samples
ens_size=$N_ENS
mean=75
std_dev=1
range_limit=5

echo "Generate a truncated normal samples for F10.7"
# Note that some versions of Singularity ASSUME
# startup in $HOME while others assume startup
# in $PWD. Need to be explicit with --pwd for 
# portability. Provide absolute path to running 
# executable and bind mount it in case.
singularity exec --bind ${TGCMMODEL} --pwd "$PWD" TIEGCM.sif /opt/miniconda3/bin/python ${TGCMMODEL}/script/generate_F107_samples.py $ens_size $mean $std_dev $range_limit

```

### Run the TIEGCM ensemble

Launch each ensemble member; this step is dependent on the creation of the ensemble member working directories

In [ ]:
# create the model directory that holds information on the model
model_dir = './model_dir' 
mkdir_future = make_dir(model_dir)

In [ ]:
# add dvc repo
future_add = add_submodule()

In [ ]:
# add storage bucket
future_setup = dvc_setup(future_add)

In [ ]:
# initialize dvc repo
future_init = dvc_init(future_setup)

In [ ]:
# utilize and set up the initialized server for tracking 
client = MlflowClient(tracking_uri = "http://127.0.0.1:8081")
mlflow.set_tracking_uri("http://127.0.0.1:8081")

### Set up experiments for MLFlow

#### Experiment 1
In Experiment 1, we train the Digit CVAE model on multiple datasets. To create these datasets, we split the original dataset into five equal, randomized parts. After each training session, we save the weights and use them as the starting point for retraining the model on the next dataset.

In [ ]:
# provide an experiment description that will appear in the UI
experiment1_description = (
    "This is the digits forecasting project."
    "This experiment contains the digit model for randomized numbers (0-9) trained separately."
)

# provide searchable tags for the experiment
experiment1_tags = {
    "project_name": "digit-forecasting",
    "model_type": "randomzied",
    "team": "digit-ml",
    "project_quarter": "Q3-2024",
    "mlflow.note.content": experiment1_description,
}

# create the experiment and give it a unique name
digit_experiment1 = client.create_experiment(
    name="Randomize_Model", tags=experiment1_tags
)

#### Experiment 2
In Experiment 2, we train the Digit CVAE model on all digit samples simultaneously, without any subsequent retraining using the weights.

In [ ]:
# provide an experiment description that will appear in the UI
experiment2_description = (
    "This is the digits forecasting project."
    "This experiment contains the digit model for numbers (0-9) trained all together."
)

# provide searchable tags for the experiment
experiment2_tags = {
    "project_name": "digit-forecasting",
    "model_type": "all digits",
    "team": "digit-ml",
    "project_quarter": "Q3-2024",
    "mlflow.note.content": experiment2_description,
}

# create the experiment and give it a unique name
digit_experiment2 = client.create_experiment(
    name="Together_Model", tags=experiment2_tags
)

#### Experiment 3
In Experiment 3, we revisit the approach used in Experiment 1 - initializing the model with the weights from a previous training session and retraining it from there. However in this experiment, we train the Digit CVAE model sequentially on each of the 10 digits (0–9), one digit at a time. After each training session, we save the weights and use them to retrain the model on the next digit. This approach induces a 'forgetting' effect, where the model gradually loses its ability to recognize previous digits with each subsequent training session.

In [ ]:
# provide an experiment description that will appear in the UI
experiment3_description = (
    "This is the digits forecasting project."
    "This experiment contains the digit model for each of the numbers (0-9) trained separately."
)

# provide searchable tags that define characteristics of the runs that will be in this experiment
experiment3_tags = {
    "project_name": "digit-forecasting",
    "model_type": "sequential",
    "team": "digit-ml",
    "project_quarter": "Q3-2024",
    "mlflow.note.content": experiment3_description,
}

# create the experiment and give it a unique name
digit_experiment3 = client.create_experiment(
    name="Sequenced_Model", tags=experiment3_tags
)

In [ ]:
# save each of the experiment's metadata
digit_experiment1 = mlflow.set_experiment("Randomize_Model")
digit_experiment2 = mlflow.set_experiment("Together_Model")
digit_experiment3 = mlflow.set_experiment("Sequenced_Model")

### Run experiments

In [ ]:
# get data with the preprocess_data() app
mnist_digits, (x_train, Y_train), (x_test, Y_test) = preprocess_data().result()

#### Experiment 1

In [ ]:
# retraining the model n times
count = 0
build1 = []

for arr in np.array_split(mnist_digits, 5):
    count += 1
    
    if (count > 1):
        print('Launching retraining...')
        # Note that we augment the counter above
        # for the next training iteration BUT the
        # .append() operation only happens after the
        # execution of the code inside the (). This
        # means that we need to reference build[count-2]
        # (and not -1) because we haven't appended
        # the future to the future list until the
        # app is launched, so we need to use the counter
        # the corresponds to the future list before
        # the launch happens.
        enforce = build1[count-2]
    else:
        print('Launching first training...')
        enforce = 0
    
    # Launch training
    build1.append(build_train_model(inputs=[arr, digit_experiment1.experiment_id, f"rand_{count}", enforce]))
    
    # Print the future status of the launched app
    print(build1[count-1])

In [ ]:
for build in build1:
    print(build)

In [ ]:
future_execute1 = execute_dvc(inputs=[future_setup, 'experiment_1.weights.h5', '1', build1[count-1]])

In [ ]:
future_rm1 = rm_dvc(inputs=[future_execute1, 'experiment_1.weights.h5'])

#### Experiment 2

In [ ]:
# train all numbers at the same time
build2 = build_train_model(inputs=[mnist_digits, digit_experiment2.experiment_id, "all", 0])

In [ ]:
future_execute2 = execute_dvc(inputs=[future_setup, 'experiment_2.weights.h5', '2', build2])

In [ ]:
future_rm2 = rm_dvc(inputs=[future_execute2, 'experiment_2.weights.h5'])

#### Experiment 3

In [ ]:
# training one number at a time
count = 0
build3 = []

for num in np.arange(10):
    count += 1
    
    train_filter = np.where(Y_train == num)
    test_filter = np.where(Y_test == num)
    
    x_trn = x_train[train_filter]
    x_tst = x_test[test_filter]
    
    digits = np.expand_dims(np.concatenate([x_trn, x_tst], axis=0), -1).astype("float32") / 255
    
    if (count > 1):
        print('Launching retraining...')
        enforce = build3[count-2]
    else:
        print('Launching first training...')
        enforce = 0
    
    # Launch training
    build3.append(build_train_model(inputs=[digits, digit_experiment3.experiment_id, num, enforce]))
    
    # Print the future status of the launched app
    print(build3[count-1])

In [ ]:
future_execute3 = execute_dvc(inputs=[future_setup, 'experiment_3.weights.h5', '3', build3[count-1]])

In [ ]:
future_rm3 = rm_dvc(inputs=[future_execute3, 'experiment_3.weights.h5'])

#### dvc run through

## Stop Parsl

The cells above can be rerun any number of times; this will simply send more and more apps to be run by Parsl. When the workflow is truly complete, it is time to call the cleanup() command. This command runs implicitly when a `main.py` script finishes executing, but it is *not* run in a notebook unless it is explicitly called as it is below.

In [ ]:
dfk.cleanup()

## Clean up Parsl log files

In [ ]:
# This directory contains Parsl monitoring logs
! rm -rf runinfo

# This directory contains the Parsl app logs
! rm -rf {log_dir}

# This is the working directory
! rm -rf {work_dir}

## Start Parsl Monitoring - Option 3 - Post workflow manual invocation

Once the Parsl `./runinfo/monitoring.db` is created, it is possible to start Parsl Monitoring and browse the results of workflow in an offline manner.  In this scenario, `parsl-visualize` can be started on the command line provided that a Conda env with `parsl[visualize]` installed is activated. For example:
```
source pw/.miniconda3/etc/profile.d/conda.sh
conda activate base
parsl-visualize sqlite:////${HOME}/<work_dir>/runinfo/monitoring.db
```
(You may need to adjust the path to the Conda environment, its name, and the path to `monitoring.db`.)